In [1]:
from qick import *
# %matplotlib widget
%matplotlib notebook
# %matplotlib inline
import matplotlib.pyplot as plt

In [2]:
import numpy as np
from numpy.polynomial import Polynomial
import matplotlib.ticker as mtick
from matplotlib.ticker import MultipleLocator
# from tqdm import tqdm
from tqdm.notebook import tqdm
import xarray as xr

In [3]:
import os
import sys
sys.path.insert(0, '../../pattern/')
sys.path.insert(0, '../../instrument/')

In [4]:
from pathlib import Path

folder_name = Path.cwd().name
data_dir = Path("Z:/labdata/qcdlabs") / folder_name
data_dir.mkdir(parents=True, exist_ok=True)

In [6]:
sys.path.insert(0, '../../pattern/')
from helper_sweep import do_sweep

from double_conversion_mixer.instr_double_conversion_mixer import DuoMixer

lo1_address = "10.0.100.24"
lo2_address = ["10.0.100.32"]
drive = DuoMixer(lo1_address, lo2_address)
drive.set_reference(ref_source='EXT')

Connected to Valon_5015 [10.0.100.24:23].
10.0.100.32
Connected to Valon_5015 [10.0.100.32:23].


In [7]:
from xilinx_qick.class_drx import drx
from xilinx_qick.class_rox import rox
from xilinx_qick.class_sweep import sweep
from xilinx_qick.instr_xilinx_v1 import XilinxProg

xilinx_1 = XilinxProg(ip_address="10.0.100.21", mode='AveragerProgram')

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
rfsoc4x2_1 PYRO:obj_b7c2c93121574d7ca1a502bdddeac802@10.0.100.21:40295
QICK running on RFSoC4x2, software version 0.2.381

Firmware configuration (built Wed Sep  6 18:49:29 2023):

	Global clocks (MHz): tProc dispatcher timing 409.600, RF reference 491.520
	Groups of related clocks: [tProc clock, DAC tile 0], [DAC tile 2], [ADC tile 0]

	2 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - fs=9830.400 Msps, fabric=614.400 MHz
		envelope memory: 65536 complex samples (6.667 us)
		32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 Msps, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		axis_avg_buffer v1.0 (no edge counter, no weights)
		memory 16384 accumulated, 1024 decima

In [8]:
if_frequency = 3.25e9
xilinx_1.reps = int(1)
dr_ch0 = 0
ro_ch0 = 0
ro_ch1 = 1

In [9]:
mod_frequency_list = np.arange(1e6, 5e6+1, 0.5e6)

In [11]:
drive.lo1.output(1)
drive.lo2[0].output(1)

lo1_frequency = 8.45e9
set_frequency = 7.8465e9

drive.set_lo1(frequency=lo1_frequency, power=17)
drive.set_lo2(idx=0, power=17)
drive.set_frequency(idx=0, set_frequency_hz=set_frequency, if_frequency_hz = if_frequency)

for idx_mod_frequency, mod_frequency in enumerate(mod_frequency_list):
    print(mod_frequency/1e9)

    dt = 100/9830.4
    t_gen = np.arange(0, 1.6*2, dt)

    s_gen = 1 * np.exp(-1j*2*np.pi*(0/1e6) * t_gen)
    s_gen += 1 * np.exp(-1j*2*np.pi*(mod_frequency/1e6) * t_gen)
    s_gen += 1 * np.exp(-1j*2*np.pi*(-mod_frequency/1e6) * t_gen)

    dr_readout1 = drx(soc=xilinx_1.soccfg,
                     dr_ch=dr_ch0, ro_ch=ro_ch0, frequency= if_frequency / 1e6, gain=1, phase=0)

    ph_0 = 1
    dr_readout1.wave.add(name='x2', t_data=t_gen, s_data=s_gen/ph_0, idx=-1, interp_order=0)
    dr_readout1.rox.set(length=1.85, delay=0.350, sleep=3)
    xilinx_1.add(dr_readout=dr_readout1)

    n_reps = 10
    _file_name = 'test4_' + str(idx_mod_frequency) + '.zarr'
    # _file_name = 'test4_' + str(idx_mod_frequency) + '_ref.zarr'
    file_name= data_dir / _file_name

    iq_data2 = []
    for idx_rep in tqdm(range(n_reps)):
        iq_data = xilinx_1.acquire_decimated(load_pulses=True, progress=False)
        iq_data2.append(iq_data)

    ds = xr.concat(iq_data2, dim='reps')
    ds = ds.assign_coords(reps=np.arange(0,n_reps,1))
    ds.to_zarr(file_name, mode='w')

drive.lo1.output(0)
drive.lo2[0].output(0)


0.001


  0%|          | 0/10 [00:00<?, ?it/s]

0.0015


C:\Users\qcduser\anaconda3\envs\QCDLabs\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


  0%|          | 0/10 [00:00<?, ?it/s]

...

  0%|          | 0/10 [00:00<?, ?it/s]

0.005


C:\Users\qcduser\anaconda3\envs\QCDLabs\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


  0%|          | 0/10 [00:00<?, ?it/s]

C:\Users\qcduser\anaconda3\envs\QCDLabs\Lib\site-packages\zarr\api\asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [71]:
import zarr
zarr.consolidate_metadata(data_dir / "test_0.zarr")

<Group file://Z:/labdata/qcdlabs/260708_dac0/test_0.zarr>

In [73]:
s_mean_list = []
s_ref_list = []
for idx_set_frequency, set_frequency in enumerate(set_frequency_list):
    _file_name = 'test4_' + str(idx_set_frequency) + '.zarr'
    _file_name_ref = 'test4_' + str(idx_set_frequency) + '_ref.zarr'

    with xr.open_zarr(data_dir / _file_name) as f:
        iq_mat2 = f['IQ decimated'].load()

    for _rox in iq_mat2.rox:

        s_data = iq_mat2.sel(rox=_rox)
        t_exp = s_data.tx

        s_mean = np.mean(s_data.mean(dim='reps'))
        s_mean_list.append(s_mean)

    _file_name_ref = 'test_' + str(idx_set_frequency) + '_ref.zarr'

    with xr.open_zarr(data_dir / _file_name_ref) as f:
        iq_mat2 = f['IQ decimated'].load()

    for _rox in iq_mat2.rox:

        s_data = iq_mat2.sel(rox=_rox)
        t_exp = s_data.tx

        s_ref = np.mean(s_data.mean(dim='reps'))
        s_ref_list.append(s_ref)


fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

ax0 = fig.add_subplot(gs[0,0])
ax1 = fig.add_subplot(gs[1,0])

# ax0.scatter(set_frequency_list/1e9, 20*np.log10(np.abs(s_mean_list)), marker='.', s=10, color='C0')
# ax1.scatter(set_frequency_list/1e9, np.unwrap(np.angle(s_mean_list)), marker='.', s=10, color='C0')
#
# ax0.scatter(set_frequency_list/1e9, 20*np.log10(np.abs(s_ref_list)), marker='.', s=10, color='C1')
# ax1.scatter(set_frequency_list/1e9, np.unwrap(np.angle(s_ref_list)), marker='.', s=10, color='C1')

s_corrected_list = np.array(s_mean_list)/np.array(s_ref_list)

ax0.plot(set_frequency_list/1e9, 20*np.log10(np.abs(s_corrected_list)), 'o-', color='C2')
ax1.plot(set_frequency_list/1e9, np.angle(s_corrected_list), 'o-', color='C2')

ax0.vlines(7.8465, -10, 10)
ax1.vlines(7.8465, -10, 10)

plt.show()

<IPython.core.display.Javascript object>

In [74]:
np.argmin(20*np.log10(np.abs(s_corrected_list)))

45

In [75]:
set_frequency_list[45]

7846500000.0

In [37]:
fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for idx_set_frequency, set_frequency in enumerate(set_frequency_list):
    _file_name = 'test_' + str(idx_set_frequency) + '.zarr'

    with xr.open_zarr(data_dir / _file_name) as f:
        iq_mat2 = f['IQ decimated'].load()

    for _rox in iq_mat2.rox:
        if idx_set_frequency == 0:
            ax0 = fig.add_subplot(gs[0,_rox])
            ax1 = fig.add_subplot(gs[1,_rox])

        # s_ref = 1
        # s_data = np.conj(iq_mat2.sel(rox=_rox))/ s_ref

        s_data = iq_mat2.sel(rox=_rox)
        t_exp = s_data.tx

        s_mean = np.mean(s_data.mean(dim='reps'))

        abs_data = xr.apply_ufunc(np.abs, s_data)
        abs_mean = abs_data.mean(dim='reps')
        abs_std = abs_data.std(dim='reps')

        ph_mean = np.angle(s_data.mean(dim='reps'))
        ph_mean = np.unwrap(ph_mean)

        ph_data = xr.apply_ufunc(np.angle, s_data)
        R = np.abs(np.exp(1j*ph_data).mean(dim='reps'))
        ph_std = np.sqrt(-2*np.log(R))

        ax0.scatter(t_exp, abs_mean, marker='.', s=10, color='C0')
        ax0.fill_between(t_exp, abs_mean-abs_std, abs_mean+abs_std, alpha=0.3, color='C0')

        ax1.scatter(t_exp, ph_mean,  marker='.', s=10, color='C0')
        ax1.fill_between(t_exp, ph_mean-ph_std, ph_mean+ph_std, alpha=0.3, color='C0')

        ax0.set_xlim(0, t_exp[-1])
        ax1.set_xlim(0, t_exp[-1])

        #if _rox == 0:
        #    ax0.plot(t_gen, np.abs(s_gen) * 48, '-', color='C1')
        #    ax1.plot(t_gen, np.unwrap(np.angle(s_gen))-np.pi*0.9, '-', color='C1')

plt.show()

<IPython.core.display.Javascript object>

In [38]:
fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

for idx_set_frequency, set_frequency in enumerate(set_frequency_list):
    _file_name = 'test_' + str(idx_set_frequency) + '.zarr'

    with xr.open_zarr(data_dir / _file_name) as f:
        iq_mat2 = f['IQ decimated'].load()

    for _rox in iq_mat2.rox:
        if idx_set_frequency == 0:
            ax0 = fig.add_subplot(gs[0,_rox])
            ax1 = fig.add_subplot(gs[1,_rox])

        # s_ref = 1
        # s_data = np.conj(iq_mat2.sel(rox=_rox))/ s_ref

        s_data = iq_mat2.sel(rox=_rox)
        t_exp = s_data.tx

        abs_data = xr.apply_ufunc(np.abs, s_data)
        abs_mean = abs_data.mean(dim='reps')
        abs_std = abs_data.std(dim='reps')

        ph_mean = np.angle(s_data.mean(dim='reps'))
        ph_mean = np.unwrap(ph_mean)

        ph_data = xr.apply_ufunc(np.angle, s_data)
        R = np.abs(np.exp(1j*ph_data).mean(dim='reps'))
        ph_std = np.sqrt(-2*np.log(R))

        ax0.scatter(t_exp, abs_mean, marker='.', s=10)
        ax0.fill_between(t_exp, abs_mean-abs_std, abs_mean+abs_std, alpha=0.3)

        ax1.scatter(t_exp, ph_mean,  marker='.', s=10)
        ax1.fill_between(t_exp, ph_mean-ph_std, ph_mean+ph_std, alpha=0.3)

        ax0.set_xlim(0, t_exp[-1])
        ax1.set_xlim(0, t_exp[-1])

        #if _rox == 0:
        #    ax0.plot(t_gen, np.abs(s_gen) * 48, '-', color='C1')
        #    ax1.plot(t_gen, np.unwrap(np.angle(s_gen))-np.pi*0.9, '-', color='C1')

plt.show()

<IPython.core.display.Javascript object>

In [25]:


s_mean_list = []
for idx_set_frequency, set_frequency in enumerate(set_frequency_list):
    _file_name = 'test_' + str(idx_set_frequency) + '.zarr'

    with xr.open_zarr(data_dir / _file_name) as f:
        iq_mat2 = f['IQ decimated'].load()

    for _rox in iq_mat2.rox:

        s_data = iq_mat2.sel(rox=_rox)
        t_exp = s_data.tx

        s_mean = np.mean(s_data.mean(dim='reps'))
        s_mean_list.append(s_mean)

        # abs_data = xr.apply_ufunc(np.abs, s_data)
        # abs_mean = abs_data.mean(dim='reps')
        # abs_std = abs_data.std(dim='reps')
        #
        # ph_mean = np.angle(s_data.mean(dim='reps'))
        # ph_mean = np.unwrap(ph_mean)
        #
        # ph_data = xr.apply_ufunc(np.angle, s_data)
        # R = np.abs(np.exp(1j*ph_data).mean(dim='reps'))
        # ph_std = np.sqrt(-2*np.log(R))

fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 2)

ax0 = fig.add_subplot(gs[0,0])
ax1 = fig.add_subplot(gs[1,0])

ax0.scatter(set_frequency_list/1e9, np.abs(s_mean_list), marker='.', s=10, color='C0')
ax1.scatter(set_frequency_list/1e9, np.unwrap(np.angle(s_mean_list)), marker='.', s=10, color='C0')

plt.show()

<IPython.core.display.Javascript object>